In [1]:
import polars as pl 
import polars.selectors as cs
from datetime import datetime
import polars_ds as pds 
import requests 
import json 
import duckdb
from pathlib import Path
from src.utils import scrape_ticket_sections, get_available_events, scrape_match_results, scrape_eliteserien_results,create_table_after_round

In [ ]:
from importlib import reload
from src.config import ELITESERIEN_SEASONS

import src.utils as u
reload(u)

df = u.scrape_eliteserien_all_xg_for_seasons(
    seasons=[(2025,2026)],
    delay_seconds=0.01,
    season_delay_seconds=2,
)
# infer_schema_length=None scans all rows since column types can differ/be
# inconsistent (e.g. int vs float) across matches with sparse stat keys.
df = pl.DataFrame(df, infer_schema_length=None).with_columns(pl.lit(datetime.now()).alias("ingested_at"))

Fetching xG for season 2026 (ID: 2025)...


In [ ]:
from src.config import CURRENT_ELITESERIEN_SEASON, DB_PATH, SCRAPE_DELAY_SECONDS, ELITESERIEN_SEASONS
table_name = 'raw_match_statistics'
with duckdb.connect(str(DB_PATH)) as connection:
    connection.register("new_data", df)
    connection.execute(
        "CREATE TABLE IF NOT EXISTS "
        f"{table_name} AS SELECT * FROM new_data LIMIT 0"
    )

    # Different seasons can return different SofaScore stat columns, so
    # align schemas before inserting instead of relying on column position.
    existing_columns = {
        row[0]: row[1]
        for row in connection.execute(
            "SELECT column_name, data_type FROM information_schema.columns "
            "WHERE table_name = ?",
            [table_name],
        ).fetchall()
    }
    new_data_schema = connection.execute("DESCRIBE new_data").fetchall()
    for column_name, column_type, *_ in new_data_schema:
        if column_name not in existing_columns:
            connection.execute(
                f'ALTER TABLE {table_name} ADD COLUMN "{column_name}" {column_type}'
            )
        elif existing_columns[column_name] in ("INTEGER", "BIGINT") and column_type == "DOUBLE":
            # A previous batch may have had this column as all-null/whole
            # numbers and locked in an integer type, silently truncating
            # decimals (e.g. xG) on later inserts. Widen it once we see a
            # batch that actually has decimal values.
            connection.execute(
                f'ALTER TABLE {table_name} ALTER COLUMN "{column_name}" TYPE DOUBLE'
            )

    connection.execute(
        f"DELETE FROM {table_name} "
        "WHERE season IN (SELECT DISTINCT season FROM new_data)"
    )
    connection.execute(f"INSERT INTO {table_name} BY NAME SELECT * FROM new_data")


In [ ]:
# List all table names
db_path = Path("data/brann.duckdb")
con = duckdb.connect(str(db_path))
table = pl.from_arrow(con.execute("SELECT * FROM fct_goal_scorers").arrow())
table.filter(pl.col('season')>=2026).filter(pl.col('home_team').str.contains('Start'),pl.col('away_team').str.contains('Brann'))

season,date,matchday,home_team,away_team,result,scorer_team,scorer_name,ingested_at
i64,date,i64,str,str,str,str,str,datetime[μs]
2026,2026-09-13,21,"""IK Start""","""SK Brann""","""2:0""","""IK Start""","""J. Cornelius""",2026-09-14 09:38:25.066763
2026,2026-09-13,21,"""IK Start""","""SK Brann""","""2:0""","""IK Start""","""O. Diallo""",2026-09-14 09:38:25.066763


In [ ]:
con = duckdb.connect('data/brann.duckdb')
schema = con.execute(
    "SELECT column_name, data_type FROM information_schema.columns "
    "WHERE table_name = 'raw_match_statistics' "
    "ORDER BY column_name"
).fetchall()
con.close()
schema

[('away_ballpossession', 'BIGINT'),
 ('away_ballpossession_1st', 'BIGINT'),
 ('away_ballpossession_2nd', 'BIGINT'),
 ('away_bigchancecreated', 'BIGINT'),
 ('away_bigchancecreated_1st', 'BIGINT'),
 ('away_bigchancecreated_2nd', 'BIGINT'),
 ('away_cornerkicks', 'BIGINT'),
 ('away_cornerkicks_1st', 'BIGINT'),
 ('away_cornerkicks_2nd', 'BIGINT'),
 ('away_fouls', 'BIGINT'),
 ('away_fouls_1st', 'BIGINT'),
 ('away_fouls_2nd', 'BIGINT'),
 ('away_freekicks', 'BIGINT'),
 ('away_freekicks_1st', 'BIGINT'),
 ('away_freekicks_2nd', 'BIGINT'),
 ('away_goalkeepersaves', 'BIGINT'),
 ('away_goalkeepersaves_1st', 'BIGINT'),
 ('away_goalkeepersaves_2nd', 'BIGINT'),
 ('away_kilometerscovered', 'DOUBLE'),
 ('away_numberofsprints', 'BIGINT'),
 ('away_passes', 'BIGINT'),
 ('away_passes_1st', 'BIGINT'),
 ('away_passes_2nd', 'BIGINT'),
 ('away_team', 'VARCHAR'),
 ('away_totalshotsongoal', 'BIGINT'),
 ('away_totalshotsongoal_1st', 'BIGINT'),
 ('away_totalshotsongoal_2nd', 'BIGINT'),
 ('away_totaltackle', 'BIGINT

In [ ]:
# List all table names
db_path = Path("data/brann.duckdb")
con = duckdb.connect(str(db_path))
table_names = con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'main'").fetchall()
print("Available tables:")
for table in table_names:
    print(f"  - {table[0]}")
con.close()

Available tables:
  - dim_teams
  - fct_goal_scorers
  - fct_league_standings
  - fct_matches
  - fct_match_statistics
  - raw_eliteserien_goal_scorers
  - raw_eliteserien_results
  - raw_match_statistics


In [ ]:
con = duckdb.connect('data/brann.duckdb')
goal_scorers = pl.from_arrow(con.execute(
    """
    SELECT *
FROM fct_match_statistics
""").arrow())
con.close()
goal_scorers = goal_scorers.filter(pl.col('home_team').str.contains('Start')).filter(pl.col('away_team').str.contains('Brann'))
goal_scorers

season,date,matchday,home_team,away_team,result,sofascore_event_id,sofascore_url,home_xg_all,away_xg_all,home_xg_1st,away_xg_1st,home_xg_2nd,away_xg_2nd,snapshot_at,home_ballpossession,away_ballpossession,home_goalkeepersaves,away_goalkeepersaves,home_cornerkicks,away_cornerkicks,home_fouls,away_fouls,home_freekicks,away_freekicks,home_yellowcards,away_yellowcards,home_totalshotsongoal,away_totalshotsongoal,ingested_at,home_passes,away_passes,home_ballpossession_1st,away_ballpossession_1st,home_totalshotsongoal_1st,away_totalshotsongoal_1st,home_goalkeepersaves_1st,…,away_freekicks_1st,home_yellowcards_1st,away_yellowcards_1st,home_ballpossession_2nd,away_ballpossession_2nd,home_totalshotsongoal_2nd,away_totalshotsongoal_2nd,home_goalkeepersaves_2nd,away_goalkeepersaves_2nd,home_cornerkicks_2nd,away_cornerkicks_2nd,home_passes_2nd,away_passes_2nd,home_freekicks_2nd,away_freekicks_2nd,home_yellowcards_2nd,away_yellowcards_2nd,home_bigchancecreated,away_bigchancecreated,home_totaltackle,away_totaltackle,home_bigchancecreated_1st,away_bigchancecreated_1st,home_totaltackle_1st,away_totaltackle_1st,home_bigchancecreated_2nd,away_bigchancecreated_2nd,home_totaltackle_2nd,away_totaltackle_2nd,home_kilometerscovered,away_kilometerscovered,home_numberofsprints,away_numberofsprints,home_fouls_1st,away_fouls_1st,home_fouls_2nd,away_fouls_2nd
i64,date,i64,str,str,str,i64,str,f64,f64,f64,f64,f64,f64,"datetime[μs, Europe/Oslo]",i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,datetime[μs],i64,i64,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,i64,i64,i64,i64,i64,i64
2016,2016-10-30,29,"""IK Start""","""SK Brann""","""1:2""",6967620,"""https://www.sofascore.com/no/f…",null,null,null,null,null,null,2026-09-11 11:44:35.321572 CEST,59,41,3,5,8,8,9,15,15,9,1,2,19,14,2026-09-11 11:44:59.183599,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2018,2018-05-12,9,"""IK Start""","""SK Brann""","""0:1""",7679866,"""https://www.sofascore.com/no/f…",null,null,null,null,null,null,2026-09-11 11:52:28.957373 CEST,38,62,5,2,2,13,7,9,9,7,1,0,5,22,2026-09-11 11:53:10.378782,272,422,33,67,3,7,2,…,4,0,0,45,55,2,15,3,1,2,7,138,161,5,3,1,0,0,2,12,11,0,0,7,4,0,2,5,7,null,null,null,null,null,null,null,null
2020,2020-12-09,29,"""IK Start""","""SK Brann""","""1:1""",9045711,"""https://www.sofascore.com/no/f…",null,null,null,null,null,null,2026-09-11 11:55:00.045839 CEST,63,37,2,2,8,5,14,7,6,13,2,0,16,16,2026-09-11 11:59:27.922653,512,309,63,37,10,9,1,…,5,0,0,62,38,6,7,1,1,5,1,254,156,5,8,2,0,3,1,9,13,3,0,4,9,0,1,5,4,null,null,null,null,null,null,null,null
2026,2026-09-13,21,"""IK Start""","""SK Brann""","""2:0""",15260906,"""https://www.sofascore.com/no/f…",2.22,1.15,1.59,0.47,0.63,0.68,2026-09-14 09:47:11.028463 CEST,37,63,4,4,9,7,10,15,15,10,0,3,22,14,2026-09-14 09:47:41.296235,344,599,41,59,15,5,2,…,4,0,2,33,67,7,9,2,2,4,3,147,303,10,6,0,1,5,0,21,17,3,0,10,7,2,0,11,10,110.63,110.77,94,91,4,5,6,10


In [ ]:
con = duckdb.connect('data/brann.duckdb')
latest = pl.from_arrow(con.execute("""
    SELECT * FROM fct_league_standings 
""").arrow())
con.close()

latest

season,matchday,team,total_points,total_goals_for,total_goals_against,goal_difference,position
i64,i64,str,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]",i64
2026,19,"""Bodø/Glimt""",44,47,14,33,1
2026,19,"""Viking FK""",43,42,18,24,2
2026,19,"""Tromsø IL""",35,34,20,14,3
2026,19,"""Molde FK""",30,36,29,7,4
2026,19,"""SK Brann""",26,36,27,9,5
…,…,…,…,…,…,…,…
2015,1,"""Viking FK""",0,0,1,-1,12
2015,1,"""Tromsø IL""",0,0,1,-1,13
2015,1,"""Bodø/Glimt""",0,1,3,-2,14


In [ ]:


# Connect to DuckDB and read raw_eliteserien_results
db_path = Path("data/brann.duckdb")
con = duckdb.connect(str(db_path))
eliteserien_db = pl.from_arrow(con.execute("SELECT * FROM dim_teams").arrow())
con.close()

print(f"Loaded {len(eliteserien_db)} records from DuckDB")
eliteserien_db

Loaded 192 records from DuckDB


season,team_name
i64,str
2015,"""Aalesunds FK"""
2015,"""Bodø/Glimt"""
2015,"""Haugesund"""
2015,"""IK Start"""
2015,"""Lillestrøm SK"""
…,…
2026,"""Sandefjord"""
2026,"""Sarpsborg 08"""
2026,"""Tromsø IL"""


In [ ]:
from src.config import ELITESERIEN_SEASONS, SCRAPE_DELAY_SECONDS
from src.utils import scrape_eliteserien_goal_scorers_for_seasons

goal_scorers = scrape_eliteserien_goal_scorers_for_seasons(
    seasons=[(2014,2015)],
    delay_seconds=1,
)

processing match 1 of 240
processing match 2 of 240
processing match 3 of 240
processing match 4 of 240
processing match 5 of 240
processing match 6 of 240
processing match 7 of 240
processing match 8 of 240
processing match 9 of 240


In [ ]:
import polars as pl
pl.DataFrame(goal_scorers)

season,date,matchday,home_team,away_team,result,scorer_team,scorer_name
i64,date,i64,str,str,str,str,str
2015,2015-04-06,1,"""Mjøndalen""","""Viking FK""","""1:0""","""Mjøndalen IF""","""S. Kapidzic"""
2015,2015-04-06,1,"""Rosenborg BK""","""Aalesunds FK""","""5:0""",""" ""","""P. Helland"""
2015,2015-04-06,1,"""Rosenborg BK""","""Aalesunds FK""","""5:0""",""" ""","""P. Helland"""
2015,2015-04-06,1,"""Rosenborg BK""","""Aalesunds FK""","""5:0""",""" ""","""A. Søderlund"""
2015,2015-04-06,1,"""Rosenborg BK""","""Aalesunds FK""","""5:0""",""" ""","""A. Søderlund"""
…,…,…,…,…,…,…,…
2026,2026-08-30,19,"""Lillestrøm SK""","""Fredrikstad FK""","""1:4""","""Lillestrøm SK""","""F. Gulbrandsen"""
2026,2026-08-30,19,"""Lillestrøm SK""","""Fredrikstad FK""","""1:4""","""Fredrikstad FK""","""S. Owusu"""
2026,2026-08-30,19,"""Lillestrøm SK""","""Fredrikstad FK""","""1:4""","""Fredrikstad FK""","""M. Nilsson"""


In [ ]:
available_events = get_available_events()

In [ ]:
for event in available_events:
    print(event['event_id'])


1085523
1187151
1188514


In [ ]:
eliteserien_results = scrape_eliteserien_results(season_id = 2025,year = 2026)

In [ ]:
pl.DataFrame(eliteserien_results)

date,matchday,home_team,away_team,result,snapshot_at
date,i64,str,str,str,"datetime[μs, UTC]"
2026-03-14,1,"""HamKam""","""Viking FK""","""2:1""",2026-09-01 09:53:45.721672 UTC
2026-03-14,1,"""Molde FK""","""Rosenborg BK""","""2:0""",2026-09-01 09:53:45.721672 UTC
2026-03-15,1,"""Kristiansund BK""","""SK Brann""","""3:2""",2026-09-01 09:53:45.721672 UTC
2026-03-15,1,"""KFUM Oslo""","""IK Start""","""2:0""",2026-09-01 09:53:45.721672 UTC
2026-03-15,1,"""Vålerenga""","""Sandefjord""","""1:0""",2026-09-01 09:53:45.721672 UTC
…,…,…,…,…,…
2026-08-30,19,"""IK Start""","""KFUM Oslo""","""4:1""",2026-09-01 09:53:45.721672 UTC
2026-08-30,19,"""Tromsø IL""","""Sarpsborg 08""","""0:0""",2026-09-01 09:53:45.721672 UTC
2026-08-30,19,"""Viking FK""","""Aalesunds FK""","""2:1""",2026-09-01 09:53:45.721672 UTC


In [ ]:
create_table_after_round(eliteserien_results)

matchday,team,points,goals_for,goals_against,goal_difference,total_points,total_goals_for,total_goals_against,total_goal_difference,table_position
i64,str,i32,i64,i64,i64,i32,i64,i64,i64,u32
1,"""HamKam""",3,2,1,1,3,2,1,1,1
1,"""KFUM Oslo""",3,2,0,2,3,2,0,2,1
1,"""Kristiansund BK""",3,3,2,1,3,3,2,1,1
1,"""Lillestrøm SK""",3,3,1,2,3,3,1,2,1
1,"""Molde FK""",3,2,0,2,3,2,0,2,1
…,…,…,…,…,…,…,…,…,…,…
18,"""KFUM Oslo""",1,1,1,0,19,19,27,-8,12
18,"""Sandefjord""",3,2,1,1,18,15,23,-8,13
18,"""Aalesunds FK""",1,5,5,0,15,27,41,-14,14


In [ ]:
paok = scrape_ticket_sections(1187151)

In [ ]:
from src.utils import scrape_eliteserien_results_for_seasons

results = scrape_eliteserien_results_for_seasons(
    seasons=[
        (2025, 2026),
        (2024, 2025),
        (2023, 2024),
    ],
    delay_seconds=5.0,
)

In [ ]:
pl.DataFrame(results).filter(pl.col('date').dt.year()==2025).filter(pl.col('home_team').str.contains('Brann'))

date,matchday,home_team,away_team,result,snapshot_at
date,i64,str,str,str,"datetime[μs, UTC]"
2025-04-06,2,"""SK Brann""","""Tromsø IL""","""3:1""",2026-09-02 11:00:27.335750 UTC
2025-04-10,17,"""SK Brann""","""Strømsgodset""","""2:1""",2026-09-02 11:00:27.335750 UTC
2025-04-27,4,"""SK Brann""","""Bryne""","""3:2""",2026-09-02 11:00:27.335750 UTC
2025-05-11,6,"""SK Brann""","""Rosenborg BK""","""0:0""",2026-09-02 11:00:27.335750 UTC
2025-05-16,7,"""SK Brann""","""Sarpsborg 08""","""2:2""",2026-09-02 11:00:27.335750 UTC
…,…,…,…,…,…
2025-09-28,19,"""SK Brann""","""Fredrikstad FK""","""1:0""",2026-09-02 11:00:27.335750 UTC
2025-10-18,25,"""SK Brann""","""Haugesund""","""4:1""",2026-09-02 11:00:27.335750 UTC
2025-10-29,23,"""SK Brann""","""Bodø/Glimt""","""1:2""",2026-09-02 11:00:27.335750 UTC


In [8]:
from src.agent import run_question

run_question("Hva er poengsnittet til Brann per måned i 2026?")

RuntimeError: SQL-spørringen kunne ikke valideres eller kjøres etter to forsøk: Spørringen bruker en tabell som agenten ikke har tilgang til.

In [13]:
import importlib
import src.agent
importlib.reload(src.agent)

<module 'src.agent' from 'c:\\Users\\Yafee Ishraq\\Documents\\brann_ticket_sales_forecasting\\src\\agent.py'>

In [14]:
for event in src.agent.agent.stream(
    {"question": "Hva er antall seire per måned for Brann i 2026"},
    stream_mode="updates",
):
    print(event)

{'plan_tasks': {'tasks': ['Finn antall seire for Brann for hver måned i 2026.'], 'task_index': 0, 'generated_sqls': [], 'collected_results': [], 'current_task': 'Finn antall seire for Brann for hver måned i 2026.', 'sql_error': '', 'attempts': 0}}
{'generate_sql': {'current_task': 'Finn antall seire for Brann for hver måned i 2026.', 'sql': "SELECT \n    EXTRACT(MONTH FROM date) AS month,\n    COUNT(*) AS wins\nFROM fct_matches\nWHERE season = 2026\n    AND (\n        (home_team = 'SK Brann' AND winner = 'home_team')\n        OR (away_team = 'SK Brann' AND winner = 'away_team')\n    )\nGROUP BY month\nORDER BY month", 'attempts': 1}}
{'execute_sql': {'columns': ['month', 'wins'], 'rows': [(4, 2), (5, 2), (7, 2), (8, 2)], 'sql_error': ''}}
{'collect_result': {'collected_results': [{'task': 'Finn antall seire for Brann for hver måned i 2026.', 'sql': "SELECT \n    EXTRACT(MONTH FROM date) AS month,\n    COUNT(*) AS wins\nFROM fct_matches\nWHERE season = 2026\n    AND (\n        (home_tea

In [9]:
import duckdb

con = duckdb.connect("data/brann.duckdb")

query = """
WITH brann_points AS (
    SELECT
        matchday,
        date,
        EXTRACT(MONTH FROM date) AS month,
        total_points
    FROM fct_league_standings
    WHERE season = 2026
      AND team = 'SK Brann'
),
brann_points_with_prev AS (
    SELECT
        month,
        total_points,
        LAG(total_points, 1, 0) OVER (ORDER BY matchday) AS prev_points
    FROM brann_points
),
brann_points_per_match AS (
    SELECT
        month,
        (total_points - prev_points) AS points_earned
    FROM brann_points_with_prev
)
SELECT
    month,
    ROUND(AVG(points_earned), 2) AS avg_points_per_match
FROM brann_points_per_match
GROUP BY month
ORDER BY month
"""

result = con.execute(query).fetchdf()
print(result)

con.close()

BinderException: Binder Error: Referenced column "date" not found in FROM clause!
Candidate bindings: "matchday", "goal_difference", "team"

LINE 5:         date,
                ^

In [ ]:
from openai import OpenAI

client = OpenAI()

for model in client.models.list().data:
    print(model.id)

gpt-4-0613
gpt-4
gpt-3.5-turbo
gpt-6-astra
gpt-realtime-2.1
gpt-realtime-2.1-mini
gpt-transcribe
gpt-live-transcribe
davinci-002
babbage-002
gpt-3.5-turbo-instruct
gpt-3.5-turbo-instruct-0914
gpt-3.5-turbo-1106
tts-1-hd
tts-1-1106
tts-1-hd-1106
text-embedding-3-small
text-embedding-3-large
gpt-3.5-turbo-0125
gpt-4-turbo
gpt-4-turbo-2024-04-09
gpt-4o
gpt-4o-2024-05-13
gpt-4o-mini-2024-07-18
gpt-4o-mini
gpt-4o-2024-08-06
omni-moderation-latest
omni-moderation-2024-09-26
o1-2024-12-17
o1
o3-mini
o3-mini-2025-01-31
gpt-4o-2024-11-20
gpt-4o-mini-search-preview-2025-03-11
gpt-4o-mini-search-preview
gpt-4o-transcribe
gpt-4o-mini-transcribe
o1-pro-2025-03-19
o1-pro
gpt-4o-mini-tts
o3-2025-04-16
o4-mini-2025-04-16
o3
o4-mini
gpt-4.1-2025-04-14
gpt-4.1
gpt-4.1-mini-2025-04-14
gpt-4.1-mini
gpt-4.1-nano-2025-04-14
gpt-4.1-nano
gpt-image-1
gpt-4o-transcribe-diarize
gpt-5-chat-latest
gpt-5-2025-08-07
gpt-5
gpt-5-mini-2025-08-07
gpt-5-mini
gpt-5-nano-2025-08-07
gpt-5-nano
gpt-audio-2025-08-28
gpt-rea

In [ ]:
client.models.list().data

[Model(id='gpt-4-0613', created=1686588896, object='model', owned_by='openai', shutdown_date='2026-10-23'),
 Model(id='gpt-4', created=1687882411, object='model', owned_by='openai', shutdown_date='2026-10-23'),
 Model(id='gpt-3.5-turbo', created=1677610602, object='model', owned_by='openai', shutdown_date='2026-10-23'),
 Model(id='gpt-6-astra', created=1787853604, object='model', owned_by='system', shutdown_date=None),
 Model(id='gpt-realtime-2.1', created=1782254687, object='model', owned_by='system', shutdown_date=None),
 Model(id='gpt-realtime-2.1-mini', created=1782254706, object='model', owned_by='system', shutdown_date=None),
 Model(id='gpt-transcribe', created=1785168027, object='model', owned_by='system', shutdown_date=None),
 Model(id='gpt-live-transcribe', created=1785168034, object='model', owned_by='system', shutdown_date=None),
 Model(id='davinci-002', created=1692634301, object='model', owned_by='system', shutdown_date='2026-09-28'),
 Model(id='babbage-002', created=16926